In [1]:
pip install -q torch torch_geometric sentence-transformers tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast
import gzip
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import transformers.utils.import_utils as _transformers_import_utils
_transformers_import_utils._torchvision_available = False
_transformers_import_utils._torchvision_version = "N/A"

from sentence_transformers import SentenceTransformer
from torch_geometric.data import HeteroData
from tqdm.auto import tqdm

In [3]:
SEQ_LEN = 15
SPLIT_MODE = "all_prefixes"  
MODEL_NAME = "sentence-transformers/sentence-t5-base"
BATCH_SIZE = 64
NORMALIZE_EMBEDDINGS = False  
SEED = 42

CANDIDATE_DATA_DIRS = [
    Path("/kaggle/input/datasets/qwerte123/hetero-data-updated-diffemb"),
    Path(r"C:\Users\Admin\Documents\GitHub\claude_plum\data"),
    Path.cwd(),
]

REQUIRED_FILES = ["sequential_data.txt", "datamaps.json", "meta.json.gz"]
DATA_DIR = next(
    (p for p in CANDIDATE_DATA_DIRS if all((p / name).exists() for name in REQUIRED_FILES)),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(f"Could not find data dir with files: {REQUIRED_FILES}")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_CACHE = OUTPUT_DIR / "sentence_t5_item_embeddings.npy"
SENTENCES_CSV = OUTPUT_DIR / "sentence_t5_item_sentences.csv"
OUTPUT_PATH = OUTPUT_DIR / "heterodata_sentence_t5.pt"

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

DATA_DIR: C:\Users\Admin\Documents\GitHub\claude_plum\data
OUTPUT_PATH: C:\Users\Admin\Documents\GitHub\claude_plum\data\heterodata_sentence_t5.pt


In [4]:
def pad_to_len(items, length, pad_value=-1):
    items = list(items)[-length:]
    return items + [pad_value] * max(0, length - len(items))


def train_test_split(path, seq_len=15, mode="all_prefixes"):
    rows = {"train": defaultdict(list), "valid": defaultdict(list), "test": defaultdict(list)}
    path = Path(path)

    with (path / "sequential_data.txt").open("r", encoding="utf-8") as file:
        for raw in file:
            ids = [int(x) for x in raw.strip().split()]
            if len(ids) < 4:
                continue
            user_id = ids[0]
            items = [x - 1 for x in ids[1:]]
            if len(items) < 3:
                continue

            train_items = items[:-2]
            valid_target = items[-2]
            test_target = items[-1]

            if mode == "legacy_one_row":
                rows["train"]["user_ID"].append(user_id)
                rows["train"]["item_ID"].append(train_items)
                rows["train"]["item_ID_next"].append(valid_target)
            elif mode == "all_prefixes":
                for end in range(1, len(train_items)):
                    rows["train"]["user_ID"].append(user_id)
                    rows["train"]["item_ID"].append(train_items[:end])
                    rows["train"]["item_ID_next"].append(train_items[end])
            else:
                raise ValueError(f"Unknown split mode: {mode}")

            rows["valid"]["user_ID"].append(user_id)
            rows["valid"]["item_ID"].append(pad_to_len(train_items, seq_len))
            rows["valid"]["item_ID_next"].append(valid_target)

            rows["test"]["user_ID"].append(user_id)
            rows["test"]["item_ID"].append(pad_to_len(items[:-1], seq_len))
            rows["test"]["item_ID_next"].append(test_target)

    return {split: pd.DataFrame(data) for split, data in rows.items()}


dfs = train_test_split(DATA_DIR, seq_len=SEQ_LEN, mode=SPLIT_MODE)
for split, df in dfs.items():
    print(split, df.shape)
    display(df.head())

train (131413, 3)


,user_ID,item_ID,item_ID_next
0,1,[0],1
1,1,"[0, 1]",2
2,2,[5],6
3,2,"[5, 6]",7
4,2,"[5, 6, 7]",8


valid (22363, 3)


,user_ID,item_ID,item_ID_next
0,1,"[0, 1, 2, -1, -1, -1, -1, -1, -1, -1, -1, -1, ...",3
1,2,"[5, 6, 7, 8, 9, -1, -1, -1, -1, -1, -1, -1, -1...",3
2,3,"[3, 11, 12, 13, 14, 15, 16, -1, -1, -1, -1, -1...",17
3,4,"[19, 20, 21, 22, -1, -1, -1, -1, -1, -1, -1, -...",3
4,5,"[3, 24, 25, 26, 27, 28, 29, -1, -1, -1, -1, -1...",30


test (22363, 3)


,user_ID,item_ID,item_ID_next
0,1,"[0, 1, 2, 3, -1, -1, -1, -1, -1, -1, -1, -1, -...",4
1,2,"[5, 6, 7, 8, 9, 3, -1, -1, -1, -1, -1, -1, -1,...",10
2,3,"[3, 11, 12, 13, 14, 15, 16, 17, -1, -1, -1, -1...",18
3,4,"[19, 20, 21, 22, 3, -1, -1, -1, -1, -1, -1, -1...",23
4,5,"[3, 24, 25, 26, 27, 28, 29, 30, -1, -1, -1, -1...",31


In [5]:
def df_to_dict_tensor(df, cols):
    res = {}
    for col in cols:
        values = df[col]
        if values.map(lambda x: isinstance(x, list)).all():
            lengths = values.map(len)
            if lengths.nunique() == 1:
                res[col] = torch.tensor(values.to_list(), dtype=torch.long)
            else:
                res[col] = [list(map(int, x)) for x in values.to_list()]
        else:
            res[col] = torch.tensor(values.to_numpy(), dtype=torch.long)

        next_col = f"{col}_next"
        if next_col in df.columns:
            res[next_col] = torch.tensor(df[next_col].to_numpy(), dtype=torch.long)

    res["user_ID"] = torch.tensor(df["user_ID"].to_numpy(), dtype=torch.long)
    return res


hist = {split: df_to_dict_tensor(df, ["item_ID"]) for split, df in dfs.items()}
print({split: {k: (type(v).__name__, getattr(v, "shape", len(v) if hasattr(v, "__len__") else None)) for k, v in data.items()} for split, data in hist.items()})

{'train': {'item_ID': ('list', 131413), 'item_ID_next': ('Tensor', torch.Size([131413])), 'user_ID': ('Tensor', torch.Size([131413]))}, 'valid': {'item_ID': ('Tensor', torch.Size([22363, 15])), 'item_ID_next': ('Tensor', torch.Size([22363])), 'user_ID': ('Tensor', torch.Size([22363]))}, 'test': {'item_ID': ('Tensor', torch.Size([22363, 15])), 'item_ID_next': ('Tensor', torch.Size([22363])), 'user_ID': ('Tensor', torch.Size([22363]))}}


In [ ]:
def parse_meta(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            yield ast.literal_eval(line)


def clean_text(value, default="Unknown"):
    if value is None:
        return default
    if isinstance(value, float) and pd.isna(value):
        return default
    text = str(value).strip()
    return text if text else default


def clean_price(value):
    if value is None:
        return "Unknown"
    if isinstance(value, float) and pd.isna(value):
        return "Unknown"
    try:
        return f"{float(value):.2f} USD"
    except (TypeError, ValueError):
        return clean_text(value)


def flatten_categories(categories):
    if not isinstance(categories, list):
        return "Unknown"
    seen = []
    for path in categories:
        if isinstance(path, list):
            for part in path:
                part = clean_text(part, default="")
                if part and part not in seen:
                    seen.append(part)
    return " > ".join(seen) if seen else "Unknown"


def build_item_sentence(row):
    return (
        f"Title: {clean_text(row.get('title'))}; "
        f"Price: {clean_price(row.get('price'))}; "
        f"Brand: {clean_text(row.get('brand'))}; "
        f"Category: {flatten_categories(row.get('categories'))}."
    )


with (DATA_DIR / "datamaps.json").open("r", encoding="utf-8") as file:
    maps = json.load(file)

asin2id = pd.DataFrame(
    {"asin": asin, "id": int(idx) - 1}
    for asin, idx in maps["item2id"].items()
)

meta = pd.DataFrame.from_records(parse_meta(DATA_DIR / "meta.json.gz"))
item_data = (
    meta.merge(asin2id, on="asin", how="inner")
        .sort_values("id")
        .reset_index(drop=True)
)

assert item_data["asin"].is_unique, "ASINs should be unique after metadata merge."
assert len(item_data) == len(maps["item2id"]), "Metadata/item2id mismatch."
assert item_data["id"].tolist() == list(range(len(item_data))), "Item rows are not aligned to zero-based item IDs."

missing_report = {
    col: int(item_data[col].isna().sum()) if col in item_data.columns else len(item_data)
    for col in ["title", "price", "brand", "categories", "related"]
}
print("Missing metadata fields:", missing_report)

item_data["category_text"] = item_data["categories"].map(flatten_categories)
item_data["sentence"] = item_data.apply(build_item_sentence, axis=1)
item_data[["id", "asin", "title", "price", "brand", "category_text", "sentence"]].head()

Missing metadata fields: {'title': 7, 'price': 585, 'brand': 2080, 'categories': 0, 'related': 82}


,id,asin,title,price,brand,category_text,sentence
0,0,B004756YJA,"OPI Nail Lacquer, Simmer and Shimmer, 0.5-Flui...",12.00,OPI,Beauty > Makeup > Nails > Nail Polish,"Title: OPI Nail Lacquer, Simmer and Shimmer, 0..."
1,1,B004ZT0SSG,OPI Red Shatter Crackle Nail Polish E55 New,3.04,OPI,Beauty > Makeup > Nails > Nail Polish,Title: OPI Red Shatter Crackle Nail Polish E55...
2,2,B0020YLEYK,SKIN79 The Prestige Beblesh Balm BB Cream Diam...,14.96,Unknown,Beauty > Skin Care > Face > Creams & Moisturizers,Title: SKIN79 The Prestige Beblesh Balm BB Cre...
3,3,7806397051,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,5.04,COKA,Beauty > Makeup > Face > Concealers & Neutrali...,Title: WAWO 15 Color Professionl Makeup Eyesha...
4,4,B002WLWX82,Dr. Scholl's Quick Heat Paraffin Spa Bath,47.95,Dr. Scholl&#39;s,Beauty > Skin Care > Hands & Nails > Paraffin ...,Title: Dr. Scholl's Quick Heat Paraffin Spa Ba...


In [7]:
item_data[["id", "asin", "sentence"]].to_csv(SENTENCES_CSV, index=False)
print(f"Saved sentences: {SENTENCES_CSV}")
print(item_data.loc[0, "sentence"])

Saved sentences: C:\Users\Admin\Documents\GitHub\claude_plum\data\sentence_t5_item_sentences.csv
Title: OPI Nail Lacquer, Simmer and Shimmer, 0.5-Fluid Ounce; Price: 12.00 USD; Brand: OPI; Category: Beauty > Makeup > Nails > Nail Polish.


## Sentence-T5 embeddings

`sentence-transformers/sentence-t5-base` maps text to a 768-dimensional dense vector. The `.npy` cache lets you rerun graph construction without recomputing embeddings.

In [8]:
if EMBEDDINGS_CACHE.exists():
    item_emb_np = np.load(EMBEDDINGS_CACHE)
    print(f"Loaded cached embeddings: {EMBEDDINGS_CACHE}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Encoding with {MODEL_NAME} on {device}")
    model = SentenceTransformer(MODEL_NAME, device=device)
    item_emb_np = model.encode(
        item_data["sentence"].tolist(),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
    )
    np.save(EMBEDDINGS_CACHE, item_emb_np)
    print(f"Saved embeddings: {EMBEDDINGS_CACHE}")

item_emb = torch.tensor(item_emb_np, dtype=torch.float32)
print("Embedding shape:", tuple(item_emb.shape), "dtype:", item_emb.dtype)
assert item_emb.shape == (len(item_data), 768), f"Expected {(len(item_data), 768)}, got {tuple(item_emb.shape)}"

Encoding with sentence-transformers/sentence-t5-base on cpu


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

c:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--sentence-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/190 [00:00<?, ?it/s]

Saved embeddings: C:\Users\Admin\Documents\GitHub\claude_plum\data\sentence_t5_item_embeddings.npy
Embedding shape: (12101, 768) dtype: torch.float32


## Related items

The original notebook used a `set`, which makes the order nondeterministic. Here related ASINs are mapped to zero-based item IDs and sorted for reproducible serialization.

In [9]:
def related_to_asins(value):
    if not isinstance(value, dict):
        return []
    out = []
    for items in value.values():
        if isinstance(items, list):
            out.extend(items)
    return out


def related_included(value):
    ids = []
    for asin in related_to_asins(value):
        mapped = maps["item2id"].get(asin)
        if mapped is not None:
            ids.append(int(mapped) - 1)
    return sorted(set(ids))


item_data["related_included"] = item_data["related"].map(related_included)
related_lengths = item_data["related_included"].map(len)
print(related_lengths.describe())
print("Items without mapped related items:", int((related_lengths == 0).sum()))

count    12101.000000
mean        24.882902
std         16.280935
min          0.000000
25%         13.000000
50%         22.000000
75%         35.000000
max         90.000000
Name: related_included, dtype: float64
Items without mapped related items: 198


## Build and save HeteroData

In [10]:
df_graph = HeteroData()
df_graph["user", "rated", "item"].history = hist

df_graph["item"].x = item_emb
df_graph["item"].text = item_data["sentence"].to_numpy(dtype=object)
df_graph["item"].asin = item_data["asin"].to_numpy(dtype=object)
df_graph["item"].category = item_data["category_text"].to_numpy(dtype=object)
df_graph["item"].related = item_data["related_included"].to_list()

gen = torch.Generator().manual_seed(SEED)
df_graph["item"].is_train = torch.rand(item_emb.shape[0], generator=gen) > 0.05

print(df_graph)
torch.save(df_graph, OUTPUT_PATH)
print(f"Saved graph: {OUTPUT_PATH}")

HeteroData(
  item={
    x=[12101, 768],
    text=[12101],
    asin=[12101],
    category=[12101],
    related=[12101],
    is_train=[12101],
  },
  (user, rated, item)={
    history={
      train={
        item_ID=[131413],
        item_ID_next=[131413],
        user_ID=[131413],
      },
      valid={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
      test={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
    },
  }
)
Saved graph: C:\Users\Admin\Documents\GitHub\claude_plum\data\heterodata_sentence_t5.pt


In [11]:
loaded = torch.load(OUTPUT_PATH, weights_only=False, map_location="cpu")
assert loaded["item"].x.shape == (len(item_data), 768)
assert len(loaded["item"].related) == len(item_data)
assert len(loaded["item"].text) == len(item_data)
for split in ["train", "valid", "test"]:
    split_hist = loaded["user", "rated", "item"].history[split]
    assert "item_ID" in split_hist and "item_ID_next" in split_hist and "user_ID" in split_hist
print("Reload OK")
loaded

Reload OK


HeteroData(
  item={
    x=[12101, 768],
    text=[12101],
    asin=[12101],
    category=[12101],
    related=[12101],
    is_train=[12101],
  },
  (user, rated, item)={
    history={
      train={
        item_ID=[131413],
        item_ID_next=[131413],
        user_ID=[131413],
      },
      valid={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
      test={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
    },
  }
)